## 01 — Live Layer Switching

Everything comes together here.

We wire three things into one map:
1. **LOD selection** — `get_lod(zoom)` picks which dataset to use
2. **Grid index culling** — the correct grid index is queried for the current viewport
3. **Event handling** — zoom and pan events trigger the right updates

After this notebook, we have a working map that automatically serves the right level of detail for any zoom and any viewport location.

## Setup — Load All LOD Files and Build All Indexes

We build one `GridIndex` per LOD level at startup. This is the one-time cost.

In [1]:
import json
import math
import time
from pathlib import Path

lod_dir = Path("../../data/lod")

LOD_FILES = {
    "coarse":     "railroads_coarse.geojson",
    "medium":     "railroads_medium.geojson",
    "fine":       "railroads_fine.geojson",
    "extra_fine": "railroads_extra_fine.geojson",
}

print("Loading LOD files...")
lod_features = {}
for name, filename in LOD_FILES.items():
    with open(lod_dir / filename) as f:
        lod_features[name] = json.load(f)["features"]
    print(f"  {name:<12} {len(lod_features[name]):>6,} features")

Loading LOD files...
  coarse        2,845 features
  medium       25,413 features
  fine         25,413 features
  extra_fine   25,413 features


In [2]:
def feature_bbox(feature):
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]


class GridIndex:
    def __init__(self, cell_size=10.0):
        self.cell_size = cell_size
        self.cells = {}

    def _cells_for_bbox(self, bbox):
        lon_min, lat_min, lon_max, lat_max = bbox
        col_min = int((lon_min + 180) / self.cell_size)
        col_max = int((lon_max + 180) / self.cell_size)
        row_min = int((lat_min +  90) / self.cell_size)
        row_max = int((lat_max +  90) / self.cell_size)
        return [
            (col, row)
            for col in range(col_min, col_max + 1)
            for row in range(row_min, row_max + 1)
        ]

    def build(self, features):
        self.cells = {}
        for idx, feature in enumerate(features):
            for cell in self._cells_for_bbox(feature_bbox(feature)):
                if cell not in self.cells:
                    self.cells[cell] = []
                self.cells[cell].append((idx, feature))

    def query(self, viewport_bbox):
        seen = set()
        results = []
        for cell in self._cells_for_bbox(viewport_bbox):
            for idx, feature in self.cells.get(cell, []):
                if idx not in seen:
                    seen.add(idx)
                    results.append(feature)
        return results

In [3]:
print("Building grid indexes...")
lod_indexes = {}
for name, features in lod_features.items():
    t0 = time.perf_counter()
    idx = GridIndex(cell_size=10.0)
    idx.build(features)
    elapsed = time.perf_counter() - t0
    lod_indexes[name] = idx
    print(f"  {name:<12} built in {elapsed:.3f}s")

print("\nAll indexes ready.")

Building grid indexes...
  coarse       built in 0.012s
  medium       built in 0.386s
  fine         built in 0.524s
  extra_fine   built in 0.193s

All indexes ready.


## The Decision and Utility Functions

In [4]:
def get_lod(zoom):
    z = int(math.floor(zoom))
    if z <= 3:  return "coarse"
    if z <= 6:  return "medium"
    if z <= 10: return "fine"
    return "extra_fine"


def leaflet_bounds_to_bbox(bounds):
    (lat_min, lon_min), (lat_max, lon_max) = bounds
    return [lon_min, lat_min, lon_max, lat_max]

## The Live Map

Two event handlers wire everything together:

- `on_zoom_change` — fires when zoom changes, switches to the correct LOD index, then re-queries
- `on_bounds_change` — fires when the user pans, re-queries the current LOD index

Both update the same layer, so there is only ever one GeoJSON layer on the map.

In [5]:
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets

# ── state ────────────────────────────────────────────────────────────────────
current_lod = get_lod(5)   # start at zoom 5

# ── map setup ────────────────────────────────────────────────────────────────
m = Map(center=[48.5, 10.0], zoom=5)

layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"color": "#cc3300", "weight": 1.5, "opacity": 0.8}
)
m.add(layer)

# ── status widgets ────────────────────────────────────────────────────────────
lod_label     = widgets.Label(value="LOD: —")
feature_label = widgets.Label(value="Features: —")
time_label    = widgets.Label(value="Query: —")
status_bar    = widgets.HBox([lod_label, feature_label, time_label])

# ── update function ───────────────────────────────────────────────────────────
def update(*args):
    global current_lod

    if not m.bounds:
        return

    # Decide which LOD to use
    new_lod = get_lod(m.zoom)
    if new_lod != current_lod:
        current_lod = new_lod

    # Query the correct grid index
    vp = leaflet_bounds_to_bbox(m.bounds)
    t0 = time.perf_counter()
    visible = lod_indexes[current_lod].query(vp)
    elapsed_ms = (time.perf_counter() - t0) * 1000

    # Update the layer
    layer.data = {"type": "FeatureCollection", "features": visible}

    # Update status
    lod_label.value     = f"LOD: {current_lod}"
    feature_label.value = f"  Features: {len(visible):,}"
    time_label.value    = f"  Query: {elapsed_ms:.2f}ms"

# ── wire events ───────────────────────────────────────────────────────────────
m.observe(update, names=["zoom", "bounds"])
update()   # initial render

widgets.VBox([m, status_bar])

**Try it:**
- Zoom out to 2 — the LOD status switches to `coarse` and feature count drops
- Zoom into a city — the LOD switches to `fine` or `extra_fine` and feature count drops (culling)
- Pan around at a fixed zoom — feature count updates as different regions come into view

The query time in the status bar shows how fast each lookup is.

## What We Just Built

Let's be explicit about the system:

```
User interaction
      │
      ▼
zoom / pan event
      │
      ├──► get_lod(zoom)  ──────► select correct GridIndex
      │
      └──► leaflet_bounds_to_bbox(bounds)
                │
                ▼
          index.query(viewport_bbox)
                │
                ▼
          visible features (deduplicated)
                │
                ▼
          update GeoJSON layer
```

Every component was built from scratch across these five modules:
- The simplified LOD files (Module 02)
- The bounding box intersection test (Module 03)
- The grid spatial index (Module 04)
- The zoom decision function (Module 05)

## Exercise A

Add a **zoom indicator** to the status bar that shows the current zoom level and a simple text label of the geographic scale (e.g. `zoom 5 — country scale`).

Use the zoom-to-scale table from Notebook 00 as a guide.

In [6]:
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets
import time

def zoom_scale_label(zoom):
    z = int(math.floor(zoom))

    if z <= 3:
        return "world scale"
    elif z <= 6:
        return "country scale"
    elif z <= 10:
        return "city/region scale"
    else:
        return "street scale"


current_lod = get_lod(5)

m = Map(center=[48.5, 10.0], zoom=5)

layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"color": "#cc3300", "weight": 1.5, "opacity": 0.8}
)
m.add(layer)

zoom_label    = widgets.Label(value="Zoom: —")
lod_label     = widgets.Label(value="LOD: —")
feature_label = widgets.Label(value="Features: —")
time_label    = widgets.Label(value="Query: —")

status_bar = widgets.HBox([zoom_label, lod_label, feature_label, time_label])

def update(*args):
    global current_lod

    if not m.bounds:
        return

    new_lod = get_lod(m.zoom)
    if new_lod != current_lod:
        current_lod = new_lod

    vp = leaflet_bounds_to_bbox(m.bounds)

    t0 = time.perf_counter()
    visible = lod_indexes[current_lod].query(vp)
    elapsed_ms = (time.perf_counter() - t0) * 1000

    layer.data = {
        "type": "FeatureCollection",
        "features": visible
    }

    zoom_label.value    = f"Zoom: {m.zoom} — {zoom_scale_label(m.zoom)}"
    lod_label.value     = f"  LOD: {current_lod}"
    feature_label.value = f"  Features: {len(visible):,}"
    time_label.value    = f"  Query: {elapsed_ms:.2f}ms"

m.observe(update, names=["zoom", "bounds"])
update()

widgets.VBox([m, status_bar])

## Exercise B

The `update()` function is called for both zoom and bounds changes — a single event handler covers both.

This means if the user zooms AND pans at the same time (which ipyleaflet reports as two rapid events), `update()` runs twice. The second call sees the final state and overwrites the first — so the result is correct, but redundant work was done.

Add a `print` statement inside `update()` that shows which property triggered the call (`zoom` or `bounds`). Then scroll/zoom the map and observe the sequence. Do zoom and bounds always fire together?

In [7]:
from ipyleaflet import Map, GeoJSON
import ipywidgets as widgets
import time
import math

def zoom_scale_label(zoom):
    z = int(math.floor(zoom))

    if z <= 3:
        return "world scale"
    elif z <= 6:
        return "country scale"
    elif z <= 10:
        return "city/region scale"
    else:
        return "street scale"


current_lod = get_lod(5)

m = Map(center=[48.5, 10.0], zoom=5)

layer = GeoJSON(
    data={"type": "FeatureCollection", "features": []},
    style={"color": "#cc3300", "weight": 1.5, "opacity": 0.8}
)
m.add(layer)

zoom_label    = widgets.Label(value="Zoom: —")
lod_label     = widgets.Label(value="LOD: —")
feature_label = widgets.Label(value="Features: —")
time_label    = widgets.Label(value="Query: —")

status_bar = widgets.HBox([zoom_label, lod_label, feature_label, time_label])

def update(change=None):
    global current_lod

    if change is not None:
        print(f"Triggered by: {change['name']}")

    if not m.bounds:
        return

    new_lod = get_lod(m.zoom)
    if new_lod != current_lod:
        current_lod = new_lod

    vp = leaflet_bounds_to_bbox(m.bounds)

    t0 = time.perf_counter()
    visible = lod_indexes[current_lod].query(vp)
    elapsed_ms = (time.perf_counter() - t0) * 1000

    layer.data = {
        "type": "FeatureCollection",
        "features": visible
    }

    zoom_label.value    = f"Zoom: {m.zoom} — {zoom_scale_label(m.zoom)}"
    lod_label.value     = f"  LOD: {current_lod}"
    feature_label.value = f"  Features: {len(visible):,}"
    time_label.value    = f"  Query: {elapsed_ms:.2f}ms"

m.observe(update, names=["zoom", "bounds"])
update()

widgets.VBox([m, status_bar])

Zoom and bounds do not always fire together, but they often happen close together.

When I zoom the map, ipyleaflet may trigger both a `zoom` change and a `bounds` change because zooming also changes the visible map area.

When I only pan the map, it usually triggers a `bounds` change without changing the zoom.

The result is still correct because the second update uses the final map state, but some redundant work can happen.

## Check Your Understanding

The map builds **four** grid indexes at startup — one per LOD level. This takes a few seconds and uses memory.

An alternative design would build the index lazily — only when the user first reaches that zoom range. Describe the tradeoffs between eager (build all at startup) and lazy (build on first use) index construction for this specific application.

---

Eager construction means all four grid indexes are built at startup.

The benefit is that once the map opens, switching between zoom levels is fast because every LOD index is already ready.

The downside is that startup takes longer and uses more memory, even if the user never zooms into some LOD levels.

Lazy construction means each grid index is built only when the user first reaches that zoom range.

The benefit is faster startup and lower memory use at the beginning.

The downside is that the first time the user zooms into a new LOD level, the map may pause while that index is being built.

For this application, eager construction is better if smooth zooming is more important. Lazy construction is better if startup speed and memory usage are more important.

## Next

In [Module 06 — Putting It All Together](../06-Putting_It_Together/README.md), we assemble a clean, well-organized viewer notebook and reflect on what we built — before seeing the library version in Module 07.